# Assignment 10 - Example

This notebook illustrates a use case for functions written in assignment 10.

In [78]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import linearmodels
from IPython.display import display


In [90]:
sp_df = pd.read_csv("data/sp500.csv.gz", parse_dates=["caldt"]).set_index("caldt")

In [91]:
sp_df

,sprtrn
caldt,
1994-01-03,-0.002165
1994-01-04,0.003115
1994-01-05,0.001414
1994-01-06,-0.000920
1994-01-07,0.005951
...,...
2021-12-27,0.013839
2021-12-28,-0.001010
2021-12-29,0.001402


In [92]:
#
# fomc_df = pd.read_csv("data/Macro Announcement Dates v2.csv", usecols=["New_FOMC"], parse_dates=["New_FOMC"]).dropna().set_index("New_FOMC")

fomc_df = pd.read_csv("data/FOMC dates.csv", parse_dates=["Date"]).dropna().set_index("Date")
fomc_df["FOMC"] = 1
fomc_df


,FOMC
Date,
1994-02-04,1
1994-03-22,1
1994-05-17,1
1994-07-06,1
1994-08-16,1
...,...
2021-06-16,1
2021-07-28,1
2021-09-22,1


In [93]:
merged_df = pd.merge(sp_df, fomc_df, how="left",
                     left_index=True, right_index=True)
merged_df['FOMC'] = merged_df['FOMC'].fillna(0)
merged_df = merged_df.sort_index()
merged_df

,sprtrn,FOMC
caldt,,
1994-01-03,-0.002165,0.0
1994-01-04,0.003115,0.0
1994-01-05,0.001414,0.0
1994-01-06,-0.000920,0.0
1994-01-07,0.005951,0.0
...,...,...
2021-12-27,0.013839,0.0
2021-12-28,-0.001010,0.0
2021-12-29,0.001402,0.0


In [115]:
reg = smf.ols(formula="sprtrn ~ FOMC", data=merged_df).fit(cov_type='HAC',
                                                           cov_kwds={'maxlags': 3},
                                                           use_t=True)
reg.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 sprtrn   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     9.596
Date:                Sun, 06 Feb 2022   Prob (F-statistic):            0.00196
Time:                        22:17:47   Log-Likelihood:                 21298.
No. Observations:                7051   AIC:                        -4.259e+04
Df Residuals:                    7049   BIC:                        -4.258e+04
Df Model:                           1                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0003      0.000      2.444      0.015    6.41e-05       0.001
FOMC           0.0024      0.001      3.098      0.002       0.001       0.004
==============================================================================
Omnibus:                     1335.487   Durbin-Watson:                   2.196
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            34527.471
Skew:                          -0.195   Prob(JB):                         0.00
Kurtosis:                      13.834   Cond. No.                         5.72
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 3 lags and without small sample correction
"""

In [117]:
print(f"Param: {reg.params['FOMC']}, t: {reg.tvalues['FOMC']}, pval: {reg.pvalues['FOMC']}")

Param: 0.0023922140408397835, t: 3.0977089793227592, pval: 0.001957896687758654


In [101]:
etp_df = pd.read_csv("data/etp_stock_timeseries3.csv", parse_dates=["Date"]).set_index(
    "Date")[["ETP Trade-Order Volume", "Stock Trade-Order Volume"]]
etp_df.columns = ["ETP_TradeOrder_Vol", "Stock_TradeOrder_Vol"]

In [102]:
etp_df

,ETP_TradeOrder_Vol,Stock_TradeOrder_Vol
Date,,
2012-01-03,0.234358,2.803817
2012-01-04,0.174360,2.891582
2012-01-05,0.169110,2.847851
2012-01-06,0.170824,2.597076
2012-01-09,0.158725,2.760884
...,...,...
2021-12-27,0.325757,2.675416
2021-12-28,0.261772,2.483439
2021-12-29,0.270935,2.469126


In [103]:
etp_fomc_df = pd.merge(etp_df, fomc_df, how="left",
                     left_index=True, right_index=True)
etp_fomc_df['FOMC'] = etp_fomc_df['FOMC'].fillna(0)
etp_fomc_df = etp_fomc_df.sort_index()

In [104]:
etp_fomc_df

,ETP_TradeOrder_Vol,Stock_TradeOrder_Vol,FOMC
Date,,,
2012-01-03,0.234358,2.803817,0.0
2012-01-04,0.174360,2.891582,0.0
2012-01-05,0.169110,2.847851,0.0
2012-01-06,0.170824,2.597076,0.0
2012-01-09,0.158725,2.760884,0.0
...,...,...,...
2021-12-27,0.325757,2.675416,0.0
2021-12-28,0.261772,2.483439,0.0
2021-12-29,0.270935,2.469126,0.0


In [107]:
reg = smf.ols("ETP_TradeOrder_Vol ~ Stock_TradeOrder_Vol * FOMC",
              data=etp_fomc_df).fit(cov_type='HAC',
                                    cov_kwds={'maxlags': 10},
                                    use_t=True)
reg.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:     ETP_TradeOrder_Vol   R-squared:                       0.080
Model:                            OLS   Adj. R-squared:                  0.079
Method:                 Least Squares   F-statistic:                     10.87
Date:                Sun, 06 Feb 2022   Prob (F-statistic):           4.32e-07
Time:                        22:07:38   Log-Likelihood:                 2341.7
No. Observations:                2516   AIC:                            -4675.
Df Residuals:                    2512   BIC:                            -4652.
Df Model:                           3                                         
Covariance Type:                  HAC                                         
=============================================================================================
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept                     0.1440      0.032      4.470      0.000       0.081       0.207
Stock_TradeOrder_Vol          0.0588      0.010      5.670      0.000       0.038       0.079
FOMC                          0.0552      0.065      0.854      0.393      -0.071       0.182
Stock_TradeOrder_Vol:FOMC    -0.0149      0.021     -0.722      0.470      -0.055       0.026
==============================================================================
Omnibus:                        3.289   Durbin-Watson:                   0.225
Prob(Omnibus):                  0.193   Jarque-Bera (JB):                3.214
Skew:                           0.074   Prob(JB):                        0.201
Kurtosis:                       3.093   Cond. No.                         127.
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 10 lags and without small sample correction
"""

In [108]:
hidden_df = pd.read_csv("data/decile_hidden_rate_stock.csv",
                        parse_dates=["Date"]).set_index("Date")
hidden_df = pd.DataFrame(hidden_df.stack(), columns=["HiddenRate"])
hidden_df

HiddenRate
Date                                      
2012-01-03 Market Cap Decile1    19.353111
           Market Cap Decile2    18.177939
           Market Cap Decile3    16.055759
           Market Cap Decile4    13.381571
           Market Cap Decile5    11.406060
...                                    ...
2021-12-31 Volatility Decile6    21.773555
           Volatility Decile7    21.686385
           Volatility Decile8    19.387051
           Volatility Decile9    19.630717
           Volatility Decile10   26.246368

[100640 rows x 1 columns]

In [109]:
oddlot_df = pd.read_csv("data/decile_oddlot_rate_stock.csv",
                        parse_dates=["Date"]).set_index("Date")
oddlot_df = pd.DataFrame(oddlot_df.stack(), columns=["OddlotRate"])
oddlot_df

OddlotRate
Date                                      
2012-01-03 Market Cap Decile1     7.485066
           Market Cap Decile2     9.765901
           Market Cap Decile3    13.531500
           Market Cap Decile4    16.589497
           Market Cap Decile5    15.667932
...                                    ...
2021-12-31 Volatility Decile6    65.187403
           Volatility Decile7    62.275215
           Volatility Decile8    54.503183
           Volatility Decile9    47.575240
           Volatility Decile10   43.364982

[100640 rows x 1 columns]

In [110]:
midas_df = pd.merge(hidden_df, oddlot_df, left_index=True, right_index=True)
midas_df

HiddenRate  OddlotRate
Date                                                  
2012-01-03 Market Cap Decile1    19.353111    7.485066
           Market Cap Decile2    18.177939    9.765901
           Market Cap Decile3    16.055759   13.531500
           Market Cap Decile4    13.381571   16.589497
           Market Cap Decile5    11.406060   15.667932
...                                    ...         ...
2021-12-31 Volatility Decile6    21.773555   65.187403
           Volatility Decile7    21.686385   62.275215
           Volatility Decile8    19.387051   54.503183
           Volatility Decile9    19.630717   47.575240
           Volatility Decile10   26.246368   43.364982

[100640 rows x 2 columns]

In [112]:
midas_df = midas_df.reset_index()
midas_df.columns = ['Date', 'Decile', 'HiddenRate', 'OddlotRate']
midas_df = midas_df.set_index(["Decile", "Date"])

In [119]:
reg = linearmodels.PanelOLS.from_formula("HiddenRate ~ OddlotRate + EntityEffects + TimeEffects", data=midas_df).fit(
    cov_type='clustered', cluster_entity=True, cluster_time=True)

reg.summary

Dep. Variable:,HiddenRate,R-squared:,0.0432
Estimator:,PanelOLS,R-squared (Between):,0.3541
No. Observations:,100640,R-squared (Within):,0.4389
Date:,"Sun, Feb 06 2022",R-squared (Overall):,0.3569
Time:,23:41:28,Log-likelihood,-1.934e+05
Cov. Estimator:,Clustered,,
,,F-statistic:,4424.9
Entities:,40,P-value,0.0000
Avg Obs:,2516.0,Distribution:,"F(1,98084)"
Min Obs:,2516.0,,
Max Obs:,2516.0,F-statistic (robust):,16.550


In [120]:
reg.params

OddlotRate    0.096679
Name: parameter, dtype: float64

In [ ]:
reg.p